# Importação das Bibliotecas

In [1]:
import pandas as pd
import numpy as np

# Acessando a base original

In [2]:
base = '../data/raw/BASE DE DADOS PEDE 2024 - DATATHON (1).xlsx'

In [3]:
excel = pd.ExcelFile(base)

print(f"Abas encontradas: {excel.sheet_names}")

Abas encontradas: ['PEDE2022', 'PEDE2023', 'PEDE2024']


In [4]:
for nome_aba in excel.sheet_names:
    df_temp = pd.read_excel(excel, sheet_name=nome_aba)
    print(f"--- ABA: {nome_aba} ---")
    print(f"Total de Colunas: {len(df_temp.columns)}")
    print(f"Colunas: {list(df_temp.columns)}")
    print("\n") 

--- ABA: PEDE2022 ---
Total de Colunas: 42
Colunas: ['RA', 'Fase', 'Turma', 'Nome', 'Ano nasc', 'Idade 22', 'Gênero', 'Ano ingresso', 'Instituição de ensino', 'Pedra 20', 'Pedra 21', 'Pedra 22', 'INDE 22', 'Cg', 'Cf', 'Ct', 'Nº Av', 'Avaliador1', 'Rec Av1', 'Avaliador2', 'Rec Av2', 'Avaliador3', 'Rec Av3', 'Avaliador4', 'Rec Av4', 'IAA', 'IEG', 'IPS', 'Rec Psicologia', 'IDA', 'Matem', 'Portug', 'Inglês', 'Indicado', 'Atingiu PV', 'IPV', 'IAN', 'Fase ideal', 'Defas', 'Destaque IEG', 'Destaque IDA', 'Destaque IPV']


--- ABA: PEDE2023 ---
Total de Colunas: 48
Colunas: ['RA', 'Fase', 'INDE 2023', 'Pedra 2023', 'Turma', 'Nome Anonimizado', 'Data de Nasc', 'Idade', 'Gênero', 'Ano ingresso', 'Instituição de ensino', 'Pedra 20', 'Pedra 21', 'Pedra 22', 'Pedra 23', 'INDE 22', 'INDE 23', 'Cg', 'Cf', 'Ct', 'Nº Av', 'Avaliador1', 'Rec Av1', 'Avaliador2', 'Rec Av2', 'Avaliador3', 'Rec Av3', 'Avaliador4', 'Rec Av4', 'IAA', 'IEG', 'IPS', 'IPP', 'Rec Psicologia', 'IDA', 'Mat', 'Por', 'Ing', 'Indicado

# Preparação das abas

In [5]:
def carregar_aba_especifica(nome_aba, mapeamento_especifico):
    df = pd.read_excel(base, sheet_name=nome_aba)
    
    df.columns = df.columns.str.strip()
    
    df = df.rename(columns=mapeamento_especifico)
    
    df['ANO_LETIVO'] = nome_aba.replace('PEDE', '')
    
    df = df.loc[:, ~df.columns.duplicated()]
    
    return df

In [6]:
map_2022 = {
    'Nome': 'Nome', 'Ano nasc': 'Data_Nasc', 'Idade 22': 'Idade',
    'INDE 22': 'INDE', 'Pedra 22': 'Pedra',
    'Matem': 'Mat', 'Portug': 'Por', 'Inglês': 'Ing',
    'Fase ideal': 'Fase_Ideal', 'Defas': 'Defasagem'
}

map_2023 = {
    'Nome Anonimizado': 'Nome', 'Data de Nasc': 'Data_Nasc',
    'INDE 2023': 'INDE', 'Pedra 2023': 'Pedra',
    'Mat': 'Mat', 'Por': 'Por', 'Ing': 'Ing',
    'Fase Ideal': 'Fase_Ideal', 'Defasagem': 'Defasagem'
}

map_2024 = {
    'Nome Anonimizado': 'Nome', 'Data de Nasc': 'Data_Nasc',
    'INDE 2024': 'INDE', 'Pedra 2024': 'Pedra',
    'Mat': 'Mat', 'Por': 'Por', 'Ing': 'Ing',
    'Fase Ideal': 'Fase_Ideal', 'Defasagem': 'Defasagem'
}

In [7]:
df_2022 = carregar_aba_especifica('PEDE2022', map_2022)
df_2023 = carregar_aba_especifica('PEDE2023', map_2023)
df_2024 = carregar_aba_especifica('PEDE2024', map_2024)

In [8]:
cols_calc = ['INDE', 'IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPV']
for c in cols_calc:
    df_2022[c] = pd.to_numeric(df_2022[c], errors='coerce').fillna(0)

In [9]:
df_2022['IPP'] = (df_2022['INDE'] -
                  (0.1*df_2022['IAN'] + 0.2*df_2022['IDA'] + 0.2*df_2022['IEG'] +
                   0.1*df_2022['IAA'] + 0.1*df_2022['IPS'] + 0.2*df_2022['IPV'])) / 0.1
df_2022['IPP'] = df_2022['IPP'].clip(0, 10).round(3)

Concatenação das abas

In [10]:
colunas_finais = [
    'RA', 'Nome', 'Fase', 'Turma', 'Idade', 'Gênero', 'Pedra', 'INDE',
    'IAA', 'IEG', 'IPS', 'IPP', 'IDA', 'Mat', 'Por', 'Ing', 
    'IPV', 'IAN', 'Atingiu PV', 'Fase_Ideal', 'Defasagem', 'ANO_LETIVO'
]

df_final = pd.concat([df_2022, df_2023, df_2024], ignore_index=True)

df_final = df_final[[c for c in colunas_finais if c in df_final.columns]]

print(f"Sucesso! Base consolidada com {len(df_final)} linhas.")
df_final.head()

Sucesso! Base consolidada com 3030 linhas.


,RA,Nome,Fase,Turma,Idade,Gênero,Pedra,INDE,IAA,IEG,...,IDA,Mat,Por,Ing,IPV,IAN,Atingiu PV,Fase_Ideal,Defasagem,ANO_LETIVO
0,RA-1,Aluno-1,7,A,19,Menina,Quartzo,5.783,8.3,4.1,...,4.0,2.7,3.5,6.0,7.278,5.0,Não,Fase 8 (Universitários),-1,2022
1,RA-2,Aluno-2,7,A,17,Menina,Ametista,7.055,8.8,5.2,...,6.8,6.3,4.5,9.7,6.778,10.0,Não,Fase 7 (3º EM),0,2022
2,RA-3,Aluno-3,7,A,17,Menina,Ágata,6.591,0.0,7.9,...,5.6,5.8,4.0,6.9,7.556,10.0,Não,Fase 7 (3º EM),0,2022
3,RA-4,Aluno-4,7,A,17,Menino,Quartzo,5.951,8.8,4.5,...,5.0,2.8,3.5,8.7,5.278,10.0,Não,Fase 7 (3º EM),0,2022
4,RA-5,Aluno-5,7,A,17,Menina,Ametista,7.427,7.9,8.6,...,5.2,7.0,2.9,5.7,7.389,10.0,Não,Fase 7 (3º EM),0,2022


# Ajustes colunas

In [11]:
cols_texto = ['Nome', 'Fase', 'Turma', 'Gênero', 'Pedra', 'IAN', 'Fase_Ideal', 'Defasagem']

for col in cols_texto:
    if col in df_final.columns:
        df_final[col] = df_final[col].astype(str).str.strip().str.title()

In [12]:
mapeamento_genero = {
    'M': 'Masculino', 'Masc': 'Masculino', 'Masculino': 'Masculino',
    'F': 'Feminino', 'Fem': 'Feminino', 'Feminino': 'Feminino'
}

df_final['Gênero'] = df_final['Gênero'].map(lambda x: mapeamento_genero.get(x, x))

In [13]:
df_final = df_final.replace('Nan', np.nan)

In [14]:
print("Categorias encontradas na coluna Pedra:")
print(df_final['Pedra'].unique())

Categorias encontradas na coluna Pedra:
['Quartzo' 'Ametista' 'Ágata' 'Topázio' 'Agata' nan 'Incluir']


In [15]:
correcao_pedras = {
    'Agata': 'Ágata',
    'Incluir': np.nan
}

df_final['Pedra'] = df_final['Pedra'].replace(correcao_pedras)

pedras_oficiais = ['Quartzo', 'Ágata', 'Ametista', 'Topázio']

df_final.loc[~df_final['Pedra'].isin(pedras_oficiais), 'Pedra'] = np.nan

In [16]:
df_final['INDE'] = pd.to_numeric(df_final['INDE'], errors='coerce').fillna(0)

In [17]:
def preencher_pedra_por_inde(linha):
    # Se a pedra já existe e é válida, mantém
    pedras_validas = ['Quartzo', 'Ágata', 'Ametista', 'Topázio']
    if pd.notna(linha['Pedra']) and linha['Pedra'] in pedras_validas:
        return linha['Pedra']
    
    # Classificação baseada no INDE
    inde = linha['INDE']
    if 2.405 <= inde <= 5.506: return 'Quartzo'
    elif 5.506 < inde <= 6.868: return 'Ágata'
    elif 6.868 < inde <= 8.230: return 'Ametista'
    elif 8.230 < inde <= 9.294: return 'Topázio'
    else: return 'Não Classificado'

In [18]:
df_final['Pedra'] = df_final.apply(preencher_pedra_por_inde, axis=1)
print('Pedras Finais')
print(df_final['Pedra'].unique())

Pedras Finais
['Quartzo' 'Ametista' 'Ágata' 'Topázio' 'Não Classificado']


In [19]:
cols_numericas = ['INDE', 'IAA', 'IEG', 'IPS', 'IPP', 'IDA', 'Mat', 'Por', 'Ing', 'IPV', 'IAN']
for col in cols_numericas:
    if col in df_final.columns:
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce').fillna(0)

In [20]:
cols_categoricas = ['Gênero', 'Fase', 'Turma', 'Fase_Ideal', 'Defasagem']
for col in cols_categoricas:
    if col in df_final.columns:
        df_final[col] = df_final[col].fillna('Não Informado')

In [21]:
print("Nulos por coluna:")
print(df_final.isnull().sum())

Nulos por coluna:
RA               0
Nome             0
Fase             0
Turma            0
Idade            0
Gênero           0
Pedra            0
INDE             0
IAA              0
IEG              0
IPS              0
IPP              0
IDA              0
Mat              0
Por              0
Ing              0
IPV              0
IAN              0
Atingiu PV    2170
Fase_Ideal       0
Defasagem        0
ANO_LETIVO       0
dtype: int64


In [22]:
mapeamento_pv = {
    'S': 'Sim', 'Sim': 'Sim',
    'N': 'Não', 'Não': 'Não'
}
df_final['Atingiu PV'] = df_final['Atingiu PV'].map(mapeamento_pv)

df_final['Atingiu PV'] = df_final['Atingiu PV'].fillna('Não')

In [23]:
print("Nulos restantes:")
print(df_final.isnull().sum())

Nulos restantes:
RA            0
Nome          0
Fase          0
Turma         0
Idade         0
Gênero        0
Pedra         0
INDE          0
IAA           0
IEG           0
IPS           0
IPP           0
IDA           0
Mat           0
Por           0
Ing           0
IPV           0
IAN           0
Atingiu PV    0
Fase_Ideal    0
Defasagem     0
ANO_LETIVO    0
dtype: int64


# Salvando base limpa

In [24]:
df_final.to_csv('../data/processed/pede_consolidado_limpo.csv', index=False)
print("\n✅ TUDO LIMPO! Arquivo salvo e pronto para a inteligência.")


✅ TUDO LIMPO! Arquivo salvo e pronto para a inteligência.
